# Embedded Temporal Feature Extraction

In [23]:
# =========================
# Feature Extraction
# =========================
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler

def build_feature_embedding_matrix(model, dataloader, device=DEV,
                                   scaler_feat=None, scaler_emb=None,
                                   fit_scalers=True):

    
    model.eval()
    model.to(device)

    all_y = []
    all_features = []
    all_embeddings = []

    with torch.no_grad():
        for batch in dataloader:
            x, y = batch
            x = x.to(device)  # (B, T, F)
            y = y.to(device)  

            # ---- embedding ----
            emb, _ = model(x)  # (B, 64)

            # ---- last timestep features ----
            last_feat = x[:, -1, :]  # (B, F)

            all_y.append(y.cpu().numpy())
            all_features.append(last_feat.cpu().numpy())
            all_embeddings.append(emb.cpu().numpy())

    y_final = np.concatenate(all_y, axis=0)
    features = np.concatenate(all_features, axis=0)
    embeddings = np.concatenate(all_embeddings, axis=0)

    # ---- initialize scalers if not provided ----
    if scaler_feat is None:
        scaler_feat = StandardScaler()
    if scaler_emb is None:
        scaler_emb = StandardScaler()

    # ---- fit or transform ----
    if fit_scalers:
        features_scaled = scaler_feat.fit_transform(features)
        emb_scaled = scaler_emb.fit_transform(embeddings)
    else:
        features_scaled = scaler_feat.transform(features)
        emb_scaled = scaler_emb.transform(embeddings)

    # ---- concatenate ----
    X_final = np.concatenate([features, embeddings], axis=1)
    #X_final = embeddings
    #X_final = features

    return X_final, y_final, scaler_feat, scaler_emb